# Script de Ingesta de Datos Meteorológicos

## Propósito

Este notebook genera archivos de ingesta de datos meteorológicos desde la API de OpenMeteo. Puede operar en dos modos:

* **`current`**: Descarga datos meteorológicos actuales (tiempo real)
* **`forecast`**: Descarga pronósticos meteorológicos

## Flujo de Ejecución

1. **Instalación de dependencias**: Librerías necesarias para conectar con OpenMeteo
2. **Configuración de parámetros**: Define modo de ingesta y usuario
3. **Carga de ubicaciones**: Lee las ubicaciones desde Unity Catalog
4. **Generación de archivos**: Descarga datos de la API y los guarda como archivos JSON

## Parámetros del Notebook

* **`mode`**: Tipo de datos a descargar (`current` o `forecast`)
* **`username`**: Usuario que ejecuta el notebook (para rutas de archivos)

---

## 1. Instalación de Dependencias

Instalamos las librerías necesarias para conectar con la API de OpenMeteo:

* **`openmeteo-requests`**: Cliente Python para la API de OpenMeteo
* **`requests-cache`**: Cacheo de peticiones HTTP para optimizar rendimiento
* **`retry-requests`**: Reintentos automáticos en caso de fallos de red

> **Nota**: Estas librerías solo se instalan en el entorno actual del notebook y no afectan otros notebooks.

In [0]:
# Instalar cliente de OpenMeteo - proporciona métodos para consultar la API
%pip install openmeteo-requests 

# Instalar librerías para optimizar peticiones HTTP:
# - requests-cache: cachea respuestas para evitar peticiones duplicadas
# - retry-requests: reintenta automáticamente peticiones fallidas
%pip install requests-cache retry-requests

## 2. Configuración de Parámetros

Creamos widgets de parámetros para hacer el notebook configurable:

### Parámetro: `mode`
* **Valores posibles**: `current` (datos actuales) o `forecast` (pronóstico)
* **Default**: `current`
* **Uso**: Define qué tipo de datos meteorológicos descargar

### Parámetro: `username`
* **Valores posibles**: Email del usuario de Databricks
* **Default**: Cadena vacía
* **Uso**: Construye la ruta donde se guardarán los archivos JSON

Estos parámetros permiten ejecutar el notebook desde un Job con diferentes configuraciones.

In [0]:
# ============================================================================
# CONFIGURACIÓN DE PARÁMETROS DEL NOTEBOOK
# ============================================================================

# Parámetro 1: mode - Define el tipo de datos meteorológicos a descargar
# Valores posibles: 'current' (tiempo actual) o 'forecast' (pronóstico)
dbutils.widgets.text('mode', 'current')
mode = dbutils.widgets.get('mode')

# Parámetro 2: username - Usuario de Databricks que ejecuta el notebook
# Se usa para construir la ruta donde se guardarán los archivos JSON
# Ejemplo: 'jaquesada92@outlook.com'
dbutils.widgets.text('username', '')
username = dbutils.widgets.get('username')

## 3. Carga de Ubicaciones y Inicialización

En este paso:

1. **Importamos la clase personalizada** `OpenMeteoWeatherIngestion` que encapsula la lógica de conexión a la API
2. **Inicializamos el objeto** pasando el usuario y `dbutils` para acceso al filesystem
3. **Cargamos las ubicaciones** desde la tabla Unity Catalog `weather.forecast_and_current_weather.locations`
4. **Convertimos a Pandas**: El DataFrame de Spark se convierte a pandas para facilitar el procesamiento iterativo

### Estructura de la tabla `locations`

La tabla contiene columnas de latitud y longitud que se usarán para consultar la API de OpenMeteo.

In [0]:
# ============================================================================
# INICIALIZACIÓN Y CARGA DE DATOS
# ============================================================================

# Importar la clase personalizada que encapsula la lógica de ingesta
# Esta clase maneja la conexión a la API de OpenMeteo y el guardado de archivos
from open_meteo_weather_ingestion import OpenMeteoWeatherIngestion

# Inicializar el objeto de ingesta
# - username: para construir rutas de archivos específicas del usuario
# - dbutils: para acceder al filesystem de Databricks (DBFS/Workspace)
weather = OpenMeteoWeatherIngestion(username, dbutils)

# Cargar ubicaciones desde Unity Catalog
# La tabla 'locations' contiene las coordenadas geográficas de los puntos de medición
# Columnas esperadas: latitude (double), longitude (double), y metadatos (city, country, etc.)
sitios_df = spark.read.table('weather.forecast_and_current_weather.locations').toPandas()


# EJECUCIÓN DE LA INGESTA DE DATOS METEOROLÓGICOS


## Ejecutar la descarga de datos desde la API de OpenMeteo

**Parámetros:**
- `sitios_df['latitude']`: Serie pandas con todas las latitudes
- `sitios_df['longitude']`: Serie pandas con todas las longitudes  
- `mode`: 'current' o 'forecast' (define el endpoint de la API)

**Comportamiento:**
1. Itera sobre cada par (latitud, longitud)
2. Consulta la API de OpenMeteo para esa ubicación
3. Guarda la respuesta JSON en:  
   `/Workspace/Users/{username}/openmeteo-databricks-pipeline/data/{mode}/`
4. Nombra los archivos con timestamp: `{timestamp}_{hash}.json`

**Manejo de errores:**
- Reintentos automáticos en caso de fallos de red (`retry-requests`)
- Cache de respuestas para evitar consultas duplicadas (`requests-cache`)

In [0]:

weather.get_weather(sitios_df['latitude'], sitios_df['longitude'], mode=mode)

## 4. Ejecución de la Ingesta

Llama al método `get_weather()` que:

1. **Itera sobre cada ubicación** (combinación latitud/longitud)
2. **Consulta la API de OpenMeteo** para esa ubicación específica
3. **Guarda la respuesta JSON** en el filesystem de Databricks

### Parámetros:
* **`latitude`**: Serie de pandas con las latitudes de todas las ubicaciones
* **`longitude`**: Serie de pandas con las longitudes de todas las ubicaciones
* **`mode`**: Define si descarga datos `current` o `forecast`

### Resultado:

Genera archivos JSON en la ruta:
```
/Workspace/Users/{username}/openmeteo-databricks-pipeline/data/{mode}/
```

Cada archivo contiene la respuesta completa de la API para una ubicación específica.